### Procesamiento de Lenguaje Natural I
# **Desafío 1**



### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

In [12]:
print(idx2word[18000])

allocate


En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


**1. Vectorizar documentos**

In [25]:
# Elijo los 5 documentos y reviso la categoría

import random
idxs = random.sample(range(11314), 5)

# Fijo los documentos para no cambiar las conclusiones en cada corrida
idxs = [4778, 8157, 3197, 2507, 3815]

In [26]:
docs = {}
for idx in idxs:
    docs[idx] = newsgroups_train.data[idx]

print("Documentos seleccionados:", idxs)
print("-" * 80)
print("")

for idx, text in docs.items():
    print("Documento:", idx)
    print("Categoría:", newsgroups_train.target_names[newsgroups_train.target[idx]])
#    print("")
#    print(text[:100])
    print("-" * 80)
#    print("")

Documentos seleccionados: [4778, 8157, 3197, 2507, 3815]
--------------------------------------------------------------------------------

Documento: 4778
Categoría: comp.sys.mac.hardware
--------------------------------------------------------------------------------
Documento: 8157
Categoría: rec.sport.baseball
--------------------------------------------------------------------------------
Documento: 3197
Categoría: misc.forsale
--------------------------------------------------------------------------------
Documento: 2507
Categoría: alt.atheism
--------------------------------------------------------------------------------
Documento: 3815
Categoría: comp.sys.mac.hardware
--------------------------------------------------------------------------------


In [27]:
# Defino la función de comparación 
# (incluye la comparación por distancia coseno e imprime 200 caracteres
# para hacer el análisis visual)

def doc_comp(corpus, idx, docs, labels):
    doc = docs[idx]
    cossim = cosine_similarity(doc, docs)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]
    print("Los documentos más similares a", idx, "son:", mostsim)
    print(idx, " -> ", corpus.target_names[labels[idx]])
    
    for simdoc_idx in mostsim:
        print(simdoc_idx, " -> ", corpus.target_names[labels[simdoc_idx]])
    print("")

    print(corpus.data[idx][:200])
    print("."*30)
    for simdoc_idx in mostsim:
        print(corpus.data[simdoc_idx][:200])
        print("."*30)
    print("")

**2. Vectorizar documentos**

In [28]:
# Vectorizo con tfidfvect y hago las comparaciones

corpus = newsgroups_train
X_train = tfidfvect.fit_transform(corpus.data)
y_train = corpus.target

for idx in idxs:
    doc_comp(corpus, idx, X_train, y_train)

Los documentos más similares a 4778 son: [ 5447  3151 10424  5653 10103]
4778  ->  comp.sys.mac.hardware
5447  ->  sci.med
3151  ->  comp.sys.mac.hardware
10424  ->  comp.sys.mac.hardware
5653  ->  comp.windows.x
10103  ->  comp.windows.x




Same in Sweden (the ergonomic keyboard is great, BUT!
the palm rests do NOT fix to the keyboard; they just sort
of rests against the table. Too bad when you have the
keyboard in your knee...

Cheer
..............................
Archive-name: typing-injury-faq/keyboards
Version: $Revision: 5.11 $ $Date: 1993/04/13 01:20:43 $

-------------------------------------------------------------------------------
      Answers To Freq
..............................


Terry, hi.  I recently bought an LCIII and a Datadesk 101E.  I can't
remember trying to rebuild the desktop with it, however it did give me
a strange problem.  When I held down shift during startup 
..............................


I am resending this message because my news program may have g

**Análisis**

Documento 4778 (comp.sys.mac.hardware): Si bien 3 de 5 categorías de los documentos similares son diferentes, en el caso del doc 5447 (sci.med) el texto habla de una lesión por uso del teclado de la computadora. Los docs 5653 y 10103 (comp.windows.x) hablan también de hardware

Documento 8157 (rec.sport.baseball): 5 de 5 coincidencias. Esta categoría parece un BOW muy específico

Documento 3197 (misc.forsale): 3 de 5 categorías son diferentes. La confusión surge de que, por tratarse de un artículo en venta, el texto describe un producto electrónico

Documento 2507 (alt.atheism): 4 de 5 hablan de ateísmo/religión. El doc 5820 (talk.politics.misc) habla de qué es ser humano, un tópico normalmente discultido en textos religiosos

Documento 3815 (comp.sys.mac.hardware): 2 de 5 coincidencias. Los documentos 9623 y 6894 no hablan de hardware y tienen alta simlaridad por azar

**Conclusión**

Si bien tiene limitantes, encontramos falsos positivos no justificados, los ejemplos muestran que la comparación documentos vectorizados parace clasificar razonablemente bien por tipología.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

In [29]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

test_cases = (X_test.shape[0])
print("Cantidad de casos en test", test_cases)

y_pred = []

for i in range(test_cases):
    cossim = cosine_similarity(X_test[i], X_train)[0]
    mostsim = np.argmax(cossim)
    pred_class = y_train[mostsim]
    y_pred.append(pred_class)

print("Clasificación Ok")




Cantidad de casos en test 7532
Clasificación Ok


In [30]:
from sklearn.metrics import classification_report

f1 = f1_score(y_test, y_pred, average="macro")
print("F1 Macro =", f1)

print(classification_report(y_test, y_pred, target_names=newsgroups_train.target_names))

F1 Macro = 0.5049911553681621
                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65      0.49

**Conclusión**

El modelo es conceptualmente parecido a un KNN con K=1. El procesamiento fue lento, infiero porque calcula la distancia coseno de cada elemento de test contra cada uno de los elementos de train. Tiene desempeño aceptable para un baseline simple (F1 = 0.51). La categoria con mayor F1 es rec.sport.hockey  y la de menor F1 es talk.religion.misc

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

X_train_text = newsgroups_train.data
y_train = newsgroups_train.target

X_test_text = newsgroups_test.data
y_test = newsgroups_test.target



pipe = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

param_grid = [
    {
        "vectorizer": [CountVectorizer(), TfidfVectorizer()],
        "vectorizer__min_df": [1, 2, 3, 5],
        "vectorizer__max_df": [0.8, 0.9, 1.0],
        "clf": [MultinomialNB(), ComplementNB()],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0]
    }
]

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_text, y_train)

print("Mejores parámetros:")
print(grid.best_params_)

print("Mejor F1 Macro en validación:")
print(grid.best_score_)

Fitting 3 folds for each of 192 candidates, totalling 576 fits


Mejores parámetros:
{'clf': ComplementNB(), 'clf__alpha': 0.1, 'vectorizer': TfidfVectorizer(), 'vectorizer__max_df': 0.8, 'vectorizer__min_df': 1}
Mejor F1 Macro en validación:
0.753480619694639


In [32]:
import pandas as pd

df_grid = pd.DataFrame(grid.cv_results_)

tabla = df_grid[[
    "param_vectorizer",
    "param_vectorizer__min_df",
    "param_vectorizer__max_df",
    "param_clf",
    "param_clf__alpha",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

tabla.head(30)

,param_vectorizer,param_vectorizer__min_df,param_vectorizer__max_df,param_clf,param_clf__alpha,mean_test_score,std_test_score,rank_test_score
132,TfidfVectorizer(),1,0.8,ComplementNB(),0.10,0.753481,0.006274,1
136,TfidfVectorizer(),1,0.9,ComplementNB(),0.10,0.753392,0.006796,2
140,TfidfVectorizer(),1,1.0,ComplementNB(),0.10,0.753392,0.006796,2
156,TfidfVectorizer(),1,0.8,ComplementNB(),0.50,0.751629,0.005433,4
160,TfidfVectorizer(),1,0.9,ComplementNB(),0.50,0.751081,0.005688,5
164,TfidfVectorizer(),1,1.0,ComplementNB(),0.50,0.751081,0.005688,5
165,TfidfVectorizer(),2,1.0,ComplementNB(),0.50,0.747245,0.006590,7
161,TfidfVectorizer(),2,0.9,ComplementNB(),0.50,0.747245,0.006590,7
157,TfidfVectorizer(),2,0.8,ComplementNB(),0.50,0.747098,0.006470,9
12,TfidfVectorizer(),1,0.8,MultinomialNB(),0.01,0.746664,0.006995,10


**Conclusiones**

En el ejercicio probamos variaciones de los siguientes modelos y parámetros:

1. *Vectorizadores*: Term frequency (CountVectorizer) y TF-IDF (TfidfVectorizer). La hipótesis es que TF-IDF debería dar mejores resultados porque es una evolución de TF (cuenta las palabras y descarta a las más comunes). Coincide con los resultados obtenidos, los 30 mejores modelos son todos TF-IDF.

2. *Frecuencia mínima para entrar al vocabulario* (vectorizer__min_df): 1, 2, 3, 5. Los mejores modelos tiene 1 (no excluyen términos poco frecuentes)

3. *Frecuencia máxima para entrar al vocabulario* (vectorizer__max_df): 0.8, 0.9, 1.0. No parece un factor importante. Los 3 mejores tienen cada uno de estos valores, con diferencias mínimas en F1.

4. *Clasificador* (clf): MultinomialNB, ComplementNB. La hipótesis es que ComplementNB va a dar mejores resultados en la clasificación de texto porque además de mirar las palabras frecuentes en cada clase tiene en cuenta las palabras frecuentes en las otras clases. Coincide con los resulatados, los 9 mejores modelos son ComplementNB.

5. *Factor de suavizado* (alpha): 0.01, 0.1, 0.5, 1.0. Factor para evitar probabilidades iguales a 0. Los mejores modelos tienen valores intermendios de suavizado (0.1 y 0.5).

**4. Transponer la matriz documento-término.**

In [33]:
x_train = tfidfvect.fit_transform(newsgroups_train.data)

# Hago la matriz traspuesta
X_words = X_train.T

word2idx = tfidfvect.vocabulary_
idx2word = {v: k for k, v in word2idx.items()}

# Defino la función de similitud por distancia coseno
def sim_words(word, X_words, word2idx, idx2word, top_n=5):
    if word not in word2idx:
        print(f"La palabra '{word}' no está en el vocabulario.")
        return
    
    idx = word2idx[word]
    cossim = cosine_similarity(X_words[idx], X_words)[0]
    mostsim = np.argsort(cossim)[::-1][1:top_n+1]
    
    print(f"Palabras más parecidas a '{word}':")
    for sim_idx in mostsim:
        print(f"  {idx2word[sim_idx]} -> {cossim[sim_idx]:.3f}")

words = ["gun", "ball", "desktop", "car", "god"]

for word in words:
    sim_words(word, X_words, word2idx, idx2word)
    print("-" * 80)

Palabras más parecidas a 'gun':
  guns -> 0.358
  crime -> 0.244
  handgun -> 0.239
  homicides -> 0.233
  firearms -> 0.233
--------------------------------------------------------------------------------
Palabras más parecidas a 'ball':
  runners -> 0.322
  relieves -> 0.303
  puzzled -> 0.293
  runner -> 0.283
  swing -> 0.280
--------------------------------------------------------------------------------
Palabras más parecidas a 'desktop':
  norton -> 0.391
  uninstall -> 0.362
  labelled -> 0.272
  folder -> 0.245
  ndw -> 0.240
--------------------------------------------------------------------------------
Palabras más parecidas a 'car':
  cars -> 0.180
  criterium -> 0.177
  civic -> 0.175
  owner -> 0.169
  dealer -> 0.168
--------------------------------------------------------------------------------
Palabras más parecidas a 'god':
  jesus -> 0.269
  bible -> 0.262
  that -> 0.256
  existence -> 0.255
  christ -> 0.251
-------------------------------------------------------

**Conclusiones**

Las palabras similares encontradas para cada uno de los 5 ejemplos tiene una vinculación clara. Exploré el caso de ndw, que no parecía claro, y significa Norton Desktop for Windows. 